# Plastic-Degrading Enzymes: PETase Analysis

This notebook demonstrates sequence and structure motif searching using PETase.

**Workflow:**
1. Query UniProt for PET-degrading enzymes
2. Search for conserved sequence motifs (catalytic residues)
3. Find the catalytic triad in 3D structures
4. Visualize the active site with py3Dmol

---

## Setup

In [ ]:
import sys
import os
import pandas as pd
import json
import glob
import py3Dmol

sys.path.insert(0, os.path.abspath('../sequence_motif'))
sys.path.insert(0, os.path.abspath('../structure_motif'))

from motif_searcher import run_motif_search
from uniprot_api import search_uniprot, to_csv
from search_3d_motif import search_single_file

MOTIF_LIBRARIES_DIR = '../sequence_motif/motif_libraries'
STRUCTURE_MOTIFS_DIR = '../structure_motif/motifs'
PROTEIN_FILES_DIR = '../protein_files'
OUTPUT_DIR = '../outputs/petase_analysis'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Setup complete!')

---
## Step 1: Retrieve PETase Sequences

Query UniProt for PET hydrolases and related cutinases.

In [ ]:
# Sample PETase sequences (known plastic-degrading enzymes)
PETASE_SEQUENCES = [
    {
        'Entry': 'A0A0K8P6T7',
        'protein_name': 'PETase (Ideonella sakaiensis)',
        'sequence': 'MNFPRASRLMQAAVLGGLMAVSAAATAQTNPYARGPNPTAASLEASAGPFTVRSFTVSRPSGYGAGTVYYPTNAGGTVGAIAIVPGYTARQSSIKWWGPRLASHGFVVITIDTNSTLDQPSSRSSQQMAALRQVASLNGTSSSPIYGKVDTARMGVMGWSMGGGGSLISAANNPSLKAAAPQAPWDSSTNFSSVTVPTLIFACENDSIAPVNSSALPIYDSMSRNAKQFLEINGGSHSCANSGNSNQALIGKKGVAWMKRFMDNDTRYSTFACENPNSTRVSDFRTANCSLEDPAANKARKEAELAAATAEQ',
        'organism': 'Ideonella sakaiensis'
    },
    {
        'Entry': 'G9BY57',
        'protein_name': 'LCC Cutinase (Leaf-branch compost)',
        'sequence': 'MKHLAAPLLLGSLALSPAAAAQTNAPYLGPNPTNAALEASSSPFSIRSFTVSRPSGYGNNTVVYPTNAGGTVGAIAIVPGYTAQKSTIKWWGPRLASHGFVVITIDTNSTLDQPSSRSSQQMAAIRQAASLNGTSSSPIYGKVDTARAGVLGWSNGGGGSLISAANNPSLKAAAPQVPWDSSTNFSSVTVPTLIFACENDSIAPVNSSALPIYDSMSRNAKQFLEINGGSHSCANSGNSNQALIGKKGVAWMKRFMDNDTRYSTFACENPNSTRVSDFRTANCSLEDPTANKARKEAELAAATAEQ',
        'organism': 'Uncultured bacterium'
    },
    {
        'Entry': 'A0A0G3B1Y0',
        'protein_name': 'Thermobifida fusca Cutinase',
        'sequence': 'MKLPVAAAALLLLAAPAAAAQTNAPYAAPNPTRASTLASPNSPFSIRSFTVSRPSGYSASLVYPTNAGGTVGAIAIVPGYTPRQSSIKWWGPRLASHGFVVITIDTNSTLDQPSSRSSQQMAAIRQVASLNGSSSSPIYGKVDTARAGVMGWSNGGGGSLISAANNPSLKAAAPQAPWDSSTNFSSVTVPTLIFACENDSIAPVNSSALPIYDSMSRNAKQFLEINGGSHSCANSGNSNQALIGKKGVAWMKRFMDNDTRYSTFACENPNSTRVSDFRTANCSLEDPAANKAR',
        'organism': 'Thermobifida fusca'
    }
]

print(f'PETase database: {len(PETASE_SEQUENCES)} sequences')
for seq in PETASE_SEQUENCES:
    print(f"  - {seq['Entry']}: {seq['protein_name']}")

In [ ]:
# Try UniProt, fall back to sample data
petase_csv = os.path.join(OUTPUT_DIR, 'petase_sequences.csv')

try:
    query = '(protein_name:"PET hydrolase" OR protein_name:cutinase) AND (reviewed:true)'
    print(f'Querying UniProt: {query}')
    data = search_uniprot(query, limit=20)
    lines = data.strip().split('\n')
    if len(lines) > 1:
        to_csv(data, petase_csv)
        petase_df = pd.read_csv(petase_csv)
        print(f'Retrieved {len(petase_df)} sequences from UniProt')
    else:
        raise ValueError('No results')
except:
    print('Using sample PETase sequences...')
    petase_df = pd.DataFrame(PETASE_SEQUENCES)
    petase_df.to_csv(petase_csv, index=False)
    print(f'Saved {len(petase_df)} sample sequences')

In [ ]:
# View sequences
petase_df[['Entry', 'Protein names']]

---
## Step 2: Sequence Motif Search

Search for conserved motifs in PETase enzymes:
- **GxSxG** - Classic serine hydrolase nucleophile motif
- **Oxyanion hole** 
- **Disulfide bonds** - Critical for thermostability

In [ ]:
# View PETase motif library
motifs_file = os.path.join(MOTIF_LIBRARIES_DIR, 'petase_motifs.csv')
petase_motifs = pd.read_csv(motifs_file)
print('PETase Motif Library:')
petase_motifs

In [ ]:
# Run sequence motif search
original_dir = os.getcwd()
os.chdir('../sequence_motif')

existing_outputs = set(glob.glob('outputs/*_all_results.csv'))

run_motif_search(
    motifs_file=motifs_file,
    motif_column='consensus',
    motif_name_column='motif_name',
    sequences_file=petase_csv,
    sequence_column='sequence',
    output_file='ignored',
    name_column='Entry'
)

new_outputs = set(glob.glob('outputs/*_all_results.csv')) - existing_outputs
seq_results_file = os.path.abspath(list(new_outputs)[0]) if new_outputs else os.path.abspath(sorted(glob.glob('outputs/*_all_results.csv'))[-1])

os.chdir(original_dir)
print(f'\nResults: {seq_results_file}')

In [ ]:
# Analyze results
seq_results = pd.read_csv(seq_results_file)
motif_counts = {}

for idx, row in seq_results.iterrows():
    if pd.notna(row.get('motifs')):
        try:
            motifs_data = json.loads(row['motifs'])
            for pattern, info in motifs_data.items():
                name = info.get('motif_name', pattern)
                count = len(info.get('matches', []))
                motif_counts[name] = motif_counts.get(name, 0) + count
        except:
            pass

print('Motif Summary:')
for name, count in sorted(motif_counts.items(), key=lambda x: -x[1]):
    print(f'  {name}: {count}')

---
## Step 3: Structure Motif Search - Catalytic Triad

Find the Ser-His-Asp catalytic triad in PETase crystal structures:
- **5XJH** - PETase
- **6EQE** - Plastic degrading polyesterase


In [ ]:
# Load PETase catalytic triad motif
motif_file = os.path.join(STRUCTURE_MOTIFS_DIR, 'petase_catalytic_triad.json')
with open(motif_file) as f:
    triad_motif = json.load(f)

print(f"Motif: {triad_motif['motif_name']}")
print(f"Description: {triad_motif['description'][:100]}...")
print(f"\nComponents:")
for c in triad_motif['components']:
    print(f"  {c['id']}: {c['residue_type']}")
print(f"\nConstraints: {len(triad_motif['constraints'])} distance constraints")

In [ ]:
# Search PETase structures
petase_pdbs = ['5XJH.pdb', '6EQE.pdb']
structure_results = {}

print('Searching for catalytic triad...')
for pdb in petase_pdbs:
    path = os.path.join(PROTEIN_FILES_DIR, pdb)
    if os.path.exists(path):
        matches = search_single_file(path, triad_motif)
        structure_results[pdb] = {'path': path, 'matches': matches, 'count': len(matches)}
        print(f"  {pdb}: {len(matches)} catalytic triad(s) found")
    else:
        print(f"  {pdb}: Not found")

---
## Step 4: py3Dmol Visualization

Visualize the PETase active site with the catalytic triad highlighted.

In [ ]:
def visualize_petase(pdb_path, matches, title):
    """Visualize PETase with catalytic triad highlighted."""
    with open(pdb_path) as f:
        pdb_data = f.read()
    
    view = py3Dmol.view(width=800, height=600)
    view.addModel(pdb_data, 'pdb')
    
    # Cartoon style - color by secondary structure
    view.setStyle({'cartoon': {'color': 'lightblue', 'opacity': 0.8}})
    
    # Highlight catalytic triad residues
    if matches:
        residues = matches[0].get('residues', [])
        colors = {'SER': 'yellow', 'HIS': 'blue', 'ASP': 'red'}
        
        for res in residues:
            color = colors.get(res['res_name'], 'green')
            view.addStyle(
                {'chain': res['chain_id'], 'resi': res['res_id']},
                {'stick': {'radius': 0.25, 'color': color},
                 'sphere': {'radius': 0.7, 'color': color, 'opacity': 0.7}}
            )
    
    # Heteroatoms (ligands)
    view.addStyle({'hetflag': True}, {'stick': {'radius': 0.2, 'color': 'magenta'}})
    
    view.zoomTo()
    print(f"\n{title}")
    print("Catalytic Triad: Yellow=SER, Blue=HIS, Red=ASP")
    return view

In [ ]:
# Visualize 5XJH - PETase
if '5XJH.pdb' in structure_results:
    data = structure_results['5XJH.pdb']
    view = visualize_petase(data['path'], data['matches'], 
                            "5XJH: Wild-type Ideonella sakaiensis PETase")
    view.show()

In [ ]:
# Visualize 6EQE
if '6EQE.pdb' in structure_results:
    data = structure_results['6EQE.pdb']
    view = visualize_petase(data['path'], data['matches'],
                            "6EQE")
    view.show()